# 09 — Multi-Head Attention

## Goal

Single-head self-attention produces one learned attention pattern for
each token position.

Multi-head attention divides the model representation into several
smaller attention heads.

Each head performs attention independently, allowing different heads to
learn different query-key matching patterns and different value
transformations.

Given a model hidden dimension

$$
C,
$$

and

$$
H
$$

attention heads, the head dimension is

$$
D = \frac{C}{H}.
$$

Therefore,

$$
C = HD.
$$

The central tensor transformation is

$$
(B,T,C)
\rightarrow
(B,H,T,D).
$$

Each head then performs the same scaled dot-product attention mechanism
studied in the previous lesson.

Finally, the head outputs are concatenated:

$$
(B,H,T,D)
\rightarrow
(B,T,C).
$$

This lesson focuses on how attention heads are represented, computed,
and recombined.

In [1]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange

## 1. Why Multiple Attention Heads?

A single attention head produces one attention distribution for each
query position.

This means that one query-key representation must capture all useful
relationships between tokens.

Multi-head attention creates several independent representation spaces.

Each head has its own query, key, and value features and therefore its
own attention matrix.

Conceptually:

<pre>
head 0 → one way of matching and retrieving information
head 1 → another way of matching and retrieving information
head 2 → another way of matching and retrieving information
...
</pre>

The heads are not assigned predefined linguistic roles. Any
specialization emerges from training.

In [2]:
batch_size: int = 2
sequence_length: int = 5
embedding_dim: int = 8
num_heads: int = 2

In [3]:
if embedding_dim % num_heads != 0:
    raise ValueError("embedding_dim must be divisible by num_heads.")

head_dim: int = embedding_dim // num_heads

print("C:", embedding_dim)
print("H:", num_heads)
print("D:", head_dim)

C: 8
H: 2
D: 4


In [5]:
x: torch.Tensor = torch.randn(
    batch_size,
    sequence_length,
    embedding_dim,
)

print(x.shape)

q: torch.Tensor = torch.randn(
    batch_size,
    sequence_length,
    embedding_dim,
)

torch.Size([2, 5, 8])


In [6]:
q_heads: torch.Tensor = rearrange(q, "b t (h d) -> b h t d", h=num_heads)
print(q.shape)
print(q_heads.shape)

torch.Size([2, 5, 8])
torch.Size([2, 2, 5, 4])


## 2. Multi-Head Query, Key, and Value Representations

The input representation has shape

$$
X \in \mathbb{R}^{B \times T \times C}.
$$

We first project it into query, key, and value representations:

$$
Q=XW_Q,
\qquad
K=XW_K,
\qquad
V=XW_V.
$$

Each projected tensor initially has shape

$$
(B,T,C).
$$

We then split the model dimension into

$$
C = H D,
$$

where

- $H$ is the number of attention heads,
- $D$ is the dimension of each head.

Using this decomposition,

$$
(B,T,C)
\rightarrow
(B,H,T,D).
$$

Each head now contains its own lower-dimensional query, key, and value
representations for all token positions.

In [7]:
query_projection = nn.Linear(
    embedding_dim,
    embedding_dim,
    bias=False,
)

key_projection = nn.Linear(
    embedding_dim,
    embedding_dim,
    bias=False,
)

value_projection = nn.Linear(
    embedding_dim,
    embedding_dim,
    bias=False,
)

q: torch.Tensor = query_projection(x)
k: torch.Tensor = key_projection(x)
v: torch.Tensor = value_projection(x)

q = rearrange(
    q,
    "b t (h d) -> b h t d",
    h=num_heads,
)

k = rearrange(
    k,
    "b t (h d) -> b h t d",
    h=num_heads,
)

v = rearrange(
    v,
    "b t (h d) -> b h t d",
    h=num_heads,
)

print("Q:", q.shape)
print("K:", k.shape)
print("V:", v.shape)

Q: torch.Size([2, 2, 5, 4])
K: torch.Size([2, 2, 5, 4])
V: torch.Size([2, 2, 5, 4])


## 3. Attention Within Each Head

Each attention head performs scaled dot-product attention independently.

For

$$
Q,K \in
\mathbb{R}^{B\times H\times T\times D},
$$

we transpose the final two dimensions of $K$:

$$
K^T
\in
\mathbb{R}^{B\times H\times D\times T}.
$$

The batched matrix multiplication

$$
QK^T
$$

therefore produces

$$
(B,H,T,D)
@
(B,H,D,T)
\rightarrow
(B,H,T,T).
$$

The dimensions of the attention score tensor mean:

<pre>
dim 0 → batch
dim 1 → attention head
dim 2 → query position
dim 3 → key position
</pre>

Therefore,

`scores[b, h, i, j]`

is the query-key compatibility score between positions `i` and `j`
inside attention head `h`.

In [8]:
scores: torch.Tensor = (q @ k.transpose(-2, -1)) / math.sqrt(head_dim)

print(scores.shape)

torch.Size([2, 2, 5, 5])


In [9]:
causal_mask: torch.Tensor = torch.tril(
    torch.ones(
        sequence_length,
        sequence_length,
        dtype=torch.bool,
        device=x.device,
    )
)

In [10]:
scores = scores.masked_fill(
    ~causal_mask,
    float("-inf"),
)

attention_weights: torch.Tensor = torch.softmax(
    scores,
    dim=-1,
)

attended: torch.Tensor = attention_weights @ v

print("weights:", attention_weights.shape)
print("attended:", attended.shape)


concatenated: torch.Tensor = rearrange(
    attended,
    "b h t d -> b t (h d)",
)

print(concatenated.shape)

weights: torch.Size([2, 2, 5, 5])
attended: torch.Size([2, 2, 5, 4])
torch.Size([2, 5, 8])


## 4. Concatenating Heads and Output Projection

Each head produces

$$
(B,T,D).
$$

After concatenating all $H$ heads,

$$
(B,H,T,D)
\rightarrow
(B,T,HD)
=
(B,T,C).
$$

The head outputs are concatenated, not averaged.

A final learned output projection

$$
W_O
\in
\mathbb{R}^{C\times C}
$$

allows information produced by different heads to be mixed before the
result is returned to the Transformer residual stream.

In [11]:
output_projection = nn.Linear(
    embedding_dim,
    embedding_dim,
    bias=False,
)

output: torch.Tensor = output_projection(concatenated)

print(output.shape)

torch.Size([2, 5, 8])


## 5. Multi-Head Causal Self-Attention Module

The complete multi-head causal self-attention computation is:

<pre>
X: (B,T,C)
        ↓
Q, K, V projections
        ↓
(B,T,C)
        ↓
split heads
        ↓
(B,H,T,D)
        ↓
scaled QKᵀ
        ↓
(B,H,T,T)
        ↓
causal mask
        ↓
softmax
        ↓
attention weights
        ↓
weighted sum of V
        ↓
(B,H,T,D)
        ↓
concatenate heads
        ↓
(B,T,C)
        ↓
output projection
        ↓
(B,T,C)
</pre>

In [12]:
class MultiHeadCausalSelfAttention(nn.Module):
    def __init__(self, embedding_dim: int, num_heads: int) -> None:
        super().__init__()

        if embedding_dim % num_heads != 0:
            raise ValueError("embedding_dim must be divisible by num_heads")

        self.embedding_dim: int = embedding_dim
        self.num_heads: int = num_heads
        self.head_dim: int = embedding_dim // num_heads

        self.query = nn.Linear(embedding_dim, embedding_dim, bias=False)
        self.key = nn.Linear(embedding_dim, embedding_dim, bias=False)
        self.value = nn.Linear(embedding_dim, embedding_dim, bias=False)
        self.output = nn.Linear(
            embedding_dim,
            embedding_dim,
            bias=False,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        sequence_length: int = x.shape[1]

        q: torch.Tensor = self.query(x)
        k: torch.Tensor = self.key(x)
        v: torch.Tensor = self.value(x)

        q = rearrange(q, "b t (h d) -> b h t d", h=self.num_heads)
        k = rearrange(k, "b t (h d) -> b h t d", h=self.num_heads)
        v = rearrange(v, "b t (h d) -> b h t d", h=self.num_heads)

        scores: torch.Tensor = (
            q @ k.transpose(-2, -1) / math.sqrt(self.head_dim)
        )

        causal_mask: torch.Tensor = torch.tril(
            torch.ones(
                sequence_length,
                sequence_length,
                dtype=torch.bool,
                device=x.device,
            )
        )

        scores = scores.masked_fill(~causal_mask, float("-inf"))

        weights: torch.Tensor = torch.softmax(scores, dim=-1)

        attended: torch.Tensor = weights @ v

        concatenated: torch.Tensor = rearrange(
            attended, "b h t d -> b t (h d)"
        )  # multi-head fusion

        return self.output(concatenated)


In [13]:
attention = MultiHeadCausalSelfAttention(
    embedding_dim=8,
    num_heads=2,
)

x: torch.Tensor = torch.randn(
    2,
    5,
    8,
)

output: torch.Tensor = attention(x)

print("input:", x.shape)
print("output:", output.shape)

input: torch.Size([2, 5, 8])
output: torch.Size([2, 5, 8])


### Model Dimension vs Attention Dimension

The model dimension $C$ and the total attention dimension do not have
to be identical.

In general,

$$
C_{\text{attn}} = HD,
$$

and the projections may be written as

$$
C \rightarrow HD.
$$

Standard multi-head attention commonly chooses

$$
HD=C,
$$

so the query, key, and value projections appear as

$$
C \rightarrow C.
$$

Each individual head still operates in a smaller space of dimension

$$
D=\frac{C}{H}.
$$

The single-head implementation in the previous lesson used a smaller
attention dimension explicitly for clarity, so its projections were

$$
C\rightarrow d
$$

and its output projection was

$$
d\rightarrow C.
$$

## Final Verification

Before closing the lesson, we verify three properties of the
multi-head attention implementation:

1. the output preserves the model shape,
2. future tokens cannot influence earlier outputs,
3. increasing the number of heads does not change the parameter count
   when the total model dimension remains fixed.

These checks validate both the tensor organization and the causal
behavior of the module.

In [14]:
embedding_dim: int = 8
num_heads: int = 2

attention = MultiHeadCausalSelfAttention(
    embedding_dim=embedding_dim,
    num_heads=num_heads,
)

x_original: torch.Tensor = torch.randn(
    1,
    5,
    embedding_dim,
)

x_modified: torch.Tensor = x_original.clone()

# Only change future positions.
x_modified[:, 3:, :] = torch.randn_like(x_modified[:, 3:, :])

with torch.no_grad():
    output_original: torch.Tensor = attention(x_original)

    output_modified: torch.Tensor = attention(x_modified)

print("input shape:", x_original.shape)
print("output shape:", output_original.shape)

print(
    "past unchanged:",
    torch.allclose(
        output_original[:, :3, :],
        output_modified[:, :3, :],
        atol=1e-6,
    ),
)

print(
    "future changed:",
    not torch.allclose(
        output_original[:, 3:, :],
        output_modified[:, 3:, :],
    ),
)

input shape: torch.Size([1, 5, 8])
output shape: torch.Size([1, 5, 8])
past unchanged: True
future changed: True


In [15]:
def count_parameters(
    model: nn.Module,
) -> int:
    return sum(parameter.numel() for parameter in model.parameters())


for num_heads in [1, 2, 4, 8]:
    attention = MultiHeadCausalSelfAttention(
        embedding_dim=8,
        num_heads=num_heads,
    )

    print(f"heads={num_heads}: {count_parameters(attention)} parameters")

heads=1: 256 parameters
heads=2: 256 parameters
heads=4: 256 parameters
heads=8: 256 parameters


## Takeaways

- Multi-head attention divides the model representation into several
  lower-dimensional attention heads.

- If the model dimension is $C$, the number of heads is $H$, and the
  head dimension is $D$, standard multi-head attention commonly uses

  $$
  C = HD.
  $$

- Query, key, and value projections first produce tensors of shape

  $$
  (B,T,C),
  $$

  which are rearranged into

  $$
  (B,H,T,D).
  $$

- Each head independently computes a causal attention matrix of shape

  $$
  (T,T).
  $$

  Across batches and heads, the full attention tensor has shape

  $$
  (B,H,T,T).
  $$

- The heads attend over the same token positions but operate in
  different learned feature subspaces.

- Head outputs are concatenated rather than averaged:

  $$
  (B,H,T,D)
  \rightarrow
  (B,T,HD).
  $$

- When $HD=C$, concatenation restores the model dimension.

- The output projection mixes information from the different heads and
  returns a tensor of shape

  $$
  (B,T,C).
  $$

- The model dimension $C$ and total attention dimension $HD$ are
  conceptually distinct, even though standard MHA often sets them equal.

- When $C$ is fixed and $HD=C$, changing the number of heads does not
  change the total size of the query, key, value, and output projection
  matrices.

- `einops.rearrange` provides a readable way to express head splitting
  and concatenation once the underlying reshape and transpose operations
  are understood.